In [ ]:
import sys, os
path = os.path.abspath('../src')
sys.path.insert(0, path)

import gedih3.gedidriver as gdr 
from gedih3.gh3builder import download_soc, build_h3db

In [ ]:
import os, sys
path = os.path.abspath('/gpfs/data1/vclgp/decontot/repos/gedih3/src')
sys.path.insert(0, path)

from gedih3.config import GEDI_BEAMS, GH3_DEFAULT_DOWNLOAD_DIR, GH3_DEFAULT_TMP_DIR, GH3_DEFAULT_SOC_DIR, GH3_DEFAULT_H3_DIR, GEDI_L2A_ESSENTIALS, GEDI_PRODUCTS, GEDI_START_DATE
from gedih3.utils import now, json_read, json_write, to_geojson, parquet_append_columns, parquet_merge_files, read_parquet_schema, h5_is_valid
from gedih3.h3utils import intersect_h3_geometries, h3_index_df, fix_h3_geometry
from gedih3.gedidriver import GEDIFile, add_special_columns, soc_file_tree, dask_h5_merged, gedi_vars_expand, gedi_vars_from_h5, validate_soc_files
from gedih3.daac import gedi_download


In [2]:
spatial = [-51,0,-50,1]
temporal = ['2020-01-01','2020-07-01']
gvars = {'L4A':['minimal'],'L4C':['minimal']}

pvars = gdr.gedi_vars_expand(gvars)
pvars

{'L4A': ['shot_number', 'agbd', 'sensitivity', 'l4_quality_flag'],
 'L4C': ['shot_number', 'wsci']}

In [3]:
gfiles = download_soc(product_vars=gvars, spatial=spatial, temporal=temporal, direct_access=True)

Successfully authenticated with Earthdata Login
No Dask client detected, using pqdm with 5 jobs
Found 27 L4A granules
Found 27 L4C granules
Found 27 L2A granules


In [4]:
ftree = gdr.soc_file_tree(gfiles, to_list=True)

In [5]:
ddf = gdr.dask_h5_merged(ftree, pvars, suffix_all=True, by_beam=True)
ddf

,agbd_l4a,sensitivity_l4a,l4_quality_flag_l4a,root_beam_l4a,root_file_l4a,wsci_l4c,root_beam_l4c,root_file_l4c,delta_time_l2a,quality_flag_l2a,lat_lowestmode_l2a,lon_lowestmode_l2a,elev_lowestmode_l2a,root_beam_l2a,root_file_l2a
npartitions=216,,,,,,,,,,,,,,,
,float32,float32,uint8,string,string,float32,string,string,float64,uint8,float64,float64,float32,string,string
,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...


In [6]:
from dask.distributed import Client

client = Client()
client

Connection method: Cluster object,Cluster type: distributed.LocalCluster
Dashboard: http://127.0.0.1:8787/status,
Dashboard: http://127.0.0.1:8787/status,Workers: 16
Total threads: 192,Total memory: 0.98 TiB
Status: running,Using processes: True
Comm: tcp://127.0.0.1:46567,Workers: 0
Dashboard: http://127.0.0.1:8787/status,Total threads: 0
Started: Just now,Total memory: 0 B
Comm: tcp://127.0.0.1:35279,Total threads: 12
Dashboard: http://127.0.0.1:37217/status,Memory: 62.93 GiB
Nanny: tcp://127.0.0.1:39301,


In [8]:
from gedih3.logger import parse_spatial
gspatial = parse_spatial(spatial)
gspatial

,geometry
0,"POLYGON ((-50 0, -50 1, -51 1, -51 0, -50 0))"


In [9]:
h3f = build_h3db(
                product_vars=pvars,
                spatial=gspatial,
                res=12,
                part=3,
                soc_source=gfiles,
                h3_dir='/gpfs/data1/vclgp/decontot/repos/gedih3/tmp/h3_s3/',
                version_kwargs={'version': 2},
            )

Listing source SOC files.
Checking for incomplete, corrupted, or existing granules to skip.
Found 27 new GEDI granules with requested products.
Indexing H3 at resolution 12, partitioning at 3.
Removing H3 partitions outside spatial filter.
Adding date and geometry columns to H3 database.
Writing partitioned H3 data to temporary directory: /gpfs/data1/vclgp/decontot/repos/gedih3/tmp/tmp
No new data to process. Finishing.
